# Dengue en Loreto (2000–2024)
## Notebook 02 · Limpieza y preparación de variables

**Entrada:** `data/processed/dengue_loreto.csv`, generado en `01_exploracion.ipynb` (107,074 casos de dengue con lugar probable de infección en Loreto).

**Qué hace este notebook:** convierte todas las edades a años, crea grupos de edad, ordena la gravedad de los casos y valida las variables antes del análisis.

**Contenido**
1. Carga de datos
2. Edad en años
3. Grupos de edad
4. Gravedad
5. Validación de sexo y semanas
6. Exportación

**Resultado:** `data/processed/dengue_loreto_limpio.csv`

---
## 1. Carga de datos

Un CSV no guarda los tipos de datos, así que en cada notebook se indica de nuevo que `ubigeo`, `edad` y `diresa` son texto. Se guarda el número inicial de filas para comprobar al final que la limpieza no eliminó ningún caso.

In [1]:
import pandas as pd
import numpy as np

loreto = pd.read_csv(
    "../data/processed/dengue_loreto.csv",
    dtype={"ubigeo": str, "edad": str, "diresa": str}
)
filas_iniciales = len(loreto)
loreto.shape

(107074, 11)

In [2]:
loreto.keys()

Index(['departamento', 'provincia', 'distrito', 'ubigeo', 'enfermedad', 'ano',
       'semana', 'diresa', 'edad', 'tipo_edad', 'sexo'],
      dtype='str')

---
## 2. Edad en años

La edad viene como texto y en tres unidades distintas, indicadas en `tipo_edad`: años (`A`), meses (`M`) y días (`D`). Un bebé de 6 meses aparece con `edad = "6"` y `tipo_edad = "M"`; sin convertirlo, contaría como si tuviera 6 años.

### 2.1 De texto a número

`errors="coerce"` convierte en `NaN` cualquier valor que no sea un número, en lugar de detener el código, para poder contar los problemas.

In [3]:
loreto["edad_num"] = pd.to_numeric(loreto["edad"], errors="coerce")
print(loreto["edad_num"].isna().sum(), "edades que no se pudieron convertir")

0 edades que no se pudieron convertir


In [4]:
loreto["edad_num"]

0         58
1         44
2         21
3         74
4         42
          ..
107069    39
107070    18
107071    53
107072     6
107073    18
Name: edad_num, Length: 107074, dtype: int64

### 2.2 Conversión a años

Cada edad se multiplica por un factor según su unidad. Un diccionario de factores es más claro y fácil de ampliar que una serie de condicionales.

In [5]:
factores = {"A": 1, "M": 1/12, "D": 1/365}

loreto["edad_anios"] = loreto["edad_num"] * loreto["tipo_edad"].map(factores)

In [6]:
loreto["edad_anios"]

0         58.0
1         44.0
2         21.0
3         74.0
4         42.0
          ... 
107069    39.0
107070    18.0
107071    53.0
107072     6.0
107073    18.0
Name: edad_anios, Length: 107074, dtype: float64

### 2.3 Validación

In [7]:
loreto["edad_anios"].describe()

count    107074.000000
mean         25.062328
std          17.916183
min           0.000000
25%          11.000000
50%          21.000000
75%          36.000000
max         104.000000
Name: edad_anios, dtype: float64

In [8]:
# Edades imposibles o muy improbables
print("Mayores de 100 años:", (loreto["edad_anios"] > 100).sum())

# Unidades raras: 12 meses o más debería registrarse en años, y 365 días o más, también
print("Meses >= 12:", ((loreto["tipo_edad"] == "M") & (loreto["edad_num"] >= 12)).sum())
print("Días >= 365:", ((loreto["tipo_edad"] == "D") & (loreto["edad_num"] >= 365)).sum())

Mayores de 100 años: 2
Meses >= 12: 1
Días >= 365: 0


**Hallazgos**

- Todas las edades se convirtieron a número sin errores.
- La edad va de **0 a 104 años**, con una **mediana de 21 años**: el dengue en Loreto afecta sobre todo a población joven.
- Solo **2 casos superan los 100 años** (máximo de 104). Son edades plausibles, así que se conservan.
- **1 caso** se registró con 12 meses o más; la conversión lo calcula correctamente, pero es una pequeña inconsistencia de registro.


---
## 3. Grupos de edad

Se usan las etapas de vida del MINSA, para que los resultados sean comparables con los reportes oficiales. `pd.cut` convierte la edad continua en categorías; con `right=False`, cada intervalo incluye su límite inferior, así que alguien de 12 años es "Adolescente" y no "Niño".

In [9]:
loreto["grupo_edad"] = pd.cut(
    loreto["edad_anios"],
    bins=[0, 12, 18, 30, 60, np.inf],
    labels=["Niño (0-11)", "Adolescente (12-17)", "Joven (18-29)", "Adulto (30-59)", "Adulto mayor (60+)"],
    right=False
)
loreto["grupo_edad"].value_counts().sort_index()

grupo_edad
Niño (0-11)            27026
Adolescente (12-17)    18689
Joven (18-29)          24942
Adulto (30-59)         30739
Adulto mayor (60+)      5678
Name: count, dtype: int64

**Hallazgos**

| Grupo | Casos | % |
|---|---|---|
| Niño (0-11) | 27,026 | 25.2% |
| Adolescente (12-17) | 18,689 | 17.5% |
| Joven (18-29) | 24,942 | 23.3% |
| Adulto (30-59) | 30,739 | 28.7% |
| Adulto mayor (60+) | 5,678 | 5.3% |

- **Más de 4 de cada 10 casos (42.7%) son menores de 18 años.**
- **Precaución:** que un grupo tenga más casos no significa que tenga más riesgo. Los adultos de 30 a 59 años son el grupo con más casos, pero también abarcan 30 años de edad. Para comparar el riesgo hace falta la población de cada grupo (incidencia), que se incorporará más adelante.

---
## 4. Gravedad

Las formas clínicas tienen un orden natural, de menos a más grave. Si no se indica, pandas las ordena alfabéticamente. Con una **variable categórica ordenada**, el orden se respeta en tablas y gráficos. Además, se acortan los nombres para que los gráficos sean legibles.

In [10]:
gravedad = {
    "DENGUE SIN SIGNOS DE ALARMA": "Sin signos de alarma",
    "DENGUE CON SIGNOS DE ALARMA": "Con signos de alarma",
    "DENGUE GRAVE": "Grave"
}

loreto["gravedad"] = pd.Categorical(
    loreto["enfermedad"].map(gravedad),
    categories=["Sin signos de alarma", "Con signos de alarma", "Grave"],
    ordered=True
)
loreto["gravedad"].value_counts(sort=False)

gravedad
Sin signos de alarma    90452
Con signos de alarma    16068
Grave                     554
Name: count, dtype: int64

In [11]:
# Variable binaria: su promedio da directamente el porcentaje de casos con signos de alarma o graves
loreto["con_alarma_o_grave"] = loreto["gravedad"] != "Sin signos de alarma"

print(f"Casos con signos de alarma o graves: {loreto['con_alarma_o_grave'].mean() * 100:.1f}%")

Casos con signos de alarma o graves: 15.5%


**Hallazgos**

| Gravedad | Loreto | Perú |
|---|---|---|
| Sin signos de alarma | 84.5% | 88.9% |
| Con signos de alarma | 15.0% | 10.7% |
| Grave | 0.5% | 0.4% |

- En Loreto, **el 15.5% de los casos tiene signos de alarma o es grave**, frente al 11.1% del total nacional.
- La diferencia puede reflejar una mayor gravedad real, pero también diferencias en cómo se clasifican o notifican los casos en cada región. Se analizará por año y por grupo de edad.

---
## 5. Validación de sexo y semanas

In [12]:
sexo = loreto["sexo"].value_counts()
pd.DataFrame({"casos": sexo, "porcentaje": (sexo / len(loreto) * 100).round(1)})

,casos,porcentaje
sexo,,
F,56143,52.4
M,50931,47.6


In [13]:
# Las semanas epidemiológicas deben ir de 1 a 52 (algunos años tienen 53)
print("Rango de semanas:", loreto["semana"].min(), "a", loreto["semana"].max())
print("Años con semana 53:", sorted(loreto.loc[loreto["semana"] == 53, "ano"].unique().tolist()))

Rango de semanas: 1 a 53
Años con semana 53: [2003, 2008, 2009, 2014, 2020]


**Hallazgos**

- Solo hay dos valores de sexo, `F` y `M`, sin vacíos ni códigos extraños. El **52.4%** de los casos son mujeres y el **47.6%**, hombres.

---
## 6. Exportación

`assert` detiene el código si la condición no se cumple: garantiza que la limpieza no eliminó ningún caso por accidente.

Se guardan solo las columnas finales: `edad_anios` y `grupo_edad` reemplazan a `edad` y `tipo_edad`, y `gravedad` reemplaza a `enfermedad`.

**Nota:** el CSV no guarda el orden de las categorías de `gravedad` ni de `grupo_edad`. Al cargarlo en los siguientes notebooks habrá que volver a definirlo.

In [14]:
# La limpieza no debe haber eliminado ningún caso
assert len(loreto) == filas_iniciales, "¡Se perdieron filas en la limpieza!"

columnas = [
    "provincia", "distrito", "ubigeo", "ano", "semana", "diresa",
    "edad_anios", "grupo_edad", "sexo", "gravedad", "con_alarma_o_grave"
]
loreto[columnas].to_csv("../data/processed/dengue_loreto_limpio.csv", index=False)

---
## Resumen

- Se convirtieron todas las edades a años, considerando sus tres unidades de registro, y se validaron.
- Se crearon **grupos de edad** según las etapas de vida del MINSA.
- Se definió la **gravedad** como variable categórica ordenada y una variable binaria de casos con signos de alarma o graves.
- Se validaron sexo y semanas epidemiológicas.
- Se generó `data/processed/dengue_loreto_limpio.csv` con **107,074 casos** y 11 columnas, sin pérdida de filas.

**Primeros hallazgos:** la mediana de edad es de 21 años, el 42.7% de los casos son menores de 18 años y el 15.5% tiene signos de alarma o es grave, más que el promedio nacional.

**Siguiente paso:** `03_sql.ipynb` (carga en una base de datos SQLite y análisis con consultas SQL).